# **MIMIC-IV Extraction Pipeline (local, DuckDB)**

This notebook re-creates the cohort and token logic of `MIMIC_IV_Analysis.ipynb`. It runs locally with DuckDB over the PhysioNet CSV files instead of BigQuery. It writes **timestamped** event tables that HTR-Former (`hto-past`) and the KerPrint-style knowledge-graph work can use directly.

* **Development:** by default it runs on the 100-patient demo (`data/raw/mimic-iv-demo-2.2`).
* **Full data:** set `HTO_RAW_DIR` (or run `extraction/run_extraction.py`). See the README.

Large tables (labs, ICU charts) are processed by DuckDB and written straight to parquet without passing through pandas, so the full data fits on a laptop. DuckDB spills to disk when it runs short of memory.

**What is kept from the Colab notebook:** the cohort definition (broad HTN, LAST HTO prescription as the index, BP on days 1–90 → median SBP/DBP → label), the `DX_<ver>_<code>` / `MED_<NAME>` tokens, the medication normalisation, and the 512-token flat sequences.

**What is deliberately different:** every feature has a time, and only events *available* at the index time are kept. Diagnoses are billed at discharge, so by default they get `time = dischtime`. This drops the index admission's diagnoses, which the Colab notebook included. Set `DX_TIME_POLICY = "admission"` to reproduce the old behaviour.

**Configuration** (every setting can be overridden with an `HTO_*` environment variable, which `run_extraction.py` uses)

In [ ]:
from pathlib import Path
import os
import json
import re
import time
import shutil
import datetime as dt
from contextlib import contextmanager

import numpy as np
import pandas as pd
import duckdb

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "extraction" else Path.cwd()


def env(name, default):
    value = os.environ.get(name)
    if value is None:
        return default
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    if isinstance(default, int) and default is not None:
        return int(value)
    return value


# ---- source data ----
RAW_DIR = Path(env("HTO_RAW_DIR", str(PROJECT_DIR / "data" / "raw" / "mimic-iv-demo-2.2"))).expanduser()
DATASET_NAME = env("HTO_DATASET", "mimic-iv-demo-2.2")

# ---- outputs ----
OUT_DIR = Path(env("HTO_OUT_DIR", str(PROJECT_DIR / "data" / "processed" / DATASET_NAME))).expanduser()
CACHE_DIR = Path(env("HTO_CACHE_DIR", str(PROJECT_DIR / "data" / "cache" / DATASET_NAME))).expanduser()
USE_PARQUET_CACHE = env("HTO_USE_PARQUET_CACHE", True)   # one-time CSV -> parquet; later runs take minutes, not hours

# ---- DuckDB resources ----
DUCKDB_MEMORY_LIMIT = env("HTO_DUCKDB_MEMORY_LIMIT", "")   # e.g. "8GB"; empty = DuckDB default (80% of RAM)
DUCKDB_THREADS = env("HTO_DUCKDB_THREADS", "")             # empty = all cores

# ---- cohort / label (same as the Colab notebook) ----
OUTCOME_WINDOW_DAYS = (1, 90)        # BP measured 1..90 days after the LAST HTO
CONTROLLED_SBP = 120                 # label 1 = median SBP < 120 AND median DBP < 80
CONTROLLED_DBP = 80

# ---- feature availability ----
DX_TIME_POLICY = env("HTO_DX_TIME_POLICY", "discharge")
                                     # "discharge": dx usable once the admission is discharged
                                     # "admission": dx usable from admittime (Colab behaviour, leaks index-admission dx)
REQUIRE_USABLE_DX = env("HTO_REQUIRE_USABLE_DX", False)
                                     # True: also require >= 1 diagnosis usable at index (drops patients whose
                                     # only diagnosed admission is the index one; shrinks the cohort)
INCLUDE_ICU_VITALS = env("HTO_INCLUDE_ICU_VITALS", True)  # ICU NIBP/ABP into the vitals table
LAB_LOOKBACK_DAYS = env("HTO_LAB_LOOKBACK_DAYS", "")      # e.g. "365" keeps only the last year of labs; empty = all
SHOW_SAMPLE_ROWS = env("HTO_SHOW_SAMPLE_ROWS", True)      # False: no patient-level rows in cell outputs, so the
                                                          # executed notebook holds aggregates only (PhysioNet DUA)

# ---- flat sequences (HTO_Eval format) ----
MAX_LEN = 512

# ---- split ----
SPLIT_SEED = 42
SPLIT_FRACS = (0.70, 0.15, 0.15)

# ---- reference numbers from the BigQuery run of MIMIC_IV_Analysis.ipynb (mimiciv_3_1), used as a sanity check ----
REFERENCE = {"mimic-iv-3.1": {"cohort": 38364, "positive_rate": 0.28}}

assert DX_TIME_POLICY in ("discharge", "admission")
LAB_LOOKBACK_DAYS = int(LAB_LOOKBACK_DAYS) if str(LAB_LOOKBACK_DAYS).strip() else None


@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    print(f"[{label}] {time.perf_counter() - start:,.1f}s")


def peek(df, n=5):
    """Show sample patient rows only when SHOW_SAMPLE_ROWS is on."""
    if SHOW_SAMPLE_ROWS:
        display(df.head(n))
    else:
        print(f"({len(df):,} rows; sample rows hidden, SHOW_SAMPLE_ROWS=False)")


for k in ["RAW_DIR", "DATASET_NAME", "OUT_DIR", "CACHE_DIR", "USE_PARQUET_CACHE", "DX_TIME_POLICY",
          "REQUIRE_USABLE_DX", "INCLUDE_ICU_VITALS", "LAB_LOOKBACK_DAYS", "SHOW_SAMPLE_ROWS"]:
    print(f"{k:20s} {globals()[k]}")

**Connecting DuckDB to the CSVs** (this replaces the BigQuery client)

Only the columns the pipeline uses are read. Every column is read as text and then cast explicitly, so type guessing on the full files can't break a long run. A missing column fails here, immediately, with its name. When the parquet cache is on, each table is converted once and later runs reuse it. ICU `chartevents` is cached only for the blood-pressure item IDs.

In [ ]:
ICU_BP_ITEMIDS = (220179, 220050, 220180, 220051)   # NIBP / arterial, systolic / diastolic

# table -> {column: type}; "CODE" = upper(trim(text)), anything else is a DuckDB type
SCHEMA = {
    "hosp": {
        "patients":        {"subject_id": "BIGINT", "gender": "VARCHAR", "anchor_age": "INTEGER", "anchor_year": "INTEGER"},
        "admissions":      {"subject_id": "BIGINT", "hadm_id": "BIGINT", "admittime": "TIMESTAMP", "dischtime": "TIMESTAMP",
                            "admission_type": "VARCHAR", "insurance": "VARCHAR", "marital_status": "VARCHAR", "race": "VARCHAR"},
        "diagnoses_icd":   {"subject_id": "BIGINT", "hadm_id": "BIGINT", "seq_num": "INTEGER",
                            "icd_code": "CODE", "icd_version": "INTEGER"},
        "d_icd_diagnoses": {"icd_code": "CODE", "icd_version": "INTEGER", "long_title": "VARCHAR"},
        "prescriptions":   {"subject_id": "BIGINT", "hadm_id": "BIGINT", "starttime": "TIMESTAMP",
                            "drug": "VARCHAR", "route": "VARCHAR", "ndc": "VARCHAR"},
        "omr":             {"subject_id": "BIGINT", "chartdate": "DATE", "result_name": "VARCHAR", "result_value": "VARCHAR"},
        "labevents":       {"subject_id": "BIGINT", "hadm_id": "BIGINT", "itemid": "INTEGER", "charttime": "TIMESTAMP",
                            "valuenum": "DOUBLE", "valueuom": "VARCHAR", "ref_range_lower": "DOUBLE",
                            "ref_range_upper": "DOUBLE", "flag": "VARCHAR"},
        "d_labitems":      {"itemid": "INTEGER", "label": "VARCHAR", "fluid": "VARCHAR", "category": "VARCHAR"},
    },
    "icu": {
        "chartevents":     {"subject_id": "BIGINT", "hadm_id": "BIGINT", "itemid": "INTEGER",
                            "charttime": "TIMESTAMP", "valuenum": "DOUBLE"},
    },
}
ROW_FILTERS = {("icu", "chartevents"): f"itemid IN {ICU_BP_ITEMIDS}"}
if not INCLUDE_ICU_VITALS:
    del SCHEMA["icu"]

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")   # lets large COPY/CTAS stream instead of buffering
if DUCKDB_MEMORY_LIMIT:
    con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
if DUCKDB_THREADS:
    con.execute(f"SET threads = {int(DUCKDB_THREADS)}")
(CACHE_DIR / "_duckdb_tmp").mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory = '{CACHE_DIR / '_duckdb_tmp'}'")


def find_csv(module, table):
    for suffix in (".csv", ".csv.gz"):
        path = RAW_DIR / module / f"{table}{suffix}"
        if path.exists():
            return path
    raise FileNotFoundError(f"{module}/{table}.csv(.gz) not found under {RAW_DIR}")


# MIMIC files are RFC-4180 CSV. Declare the dialect rather than letting DuckDB sniff it from the first rows:
# a sniff that sees no quoted field disables quoting and then fails millions of rows later on '"500,000 Unit"'.
CSV_OPTS = "header = true, all_varchar = true, delim = ',', quote = '\"', escape = '\"'"


def typed_select(module, table, path):
    header = con.execute(f"DESCRIBE SELECT * FROM read_csv('{path}', {CSV_OPTS})").df()
    missing = set(SCHEMA[module][table]) - set(header["column_name"])
    if missing:
        raise KeyError(f"{module}/{table} is missing columns {sorted(missing)}; found {list(header['column_name'])}")
    cols = []
    for c, t in SCHEMA[module][table].items():
        if t == "CODE":
            cols.append(f"upper(trim({c})) AS {c}")
        elif t in ("DOUBLE", "INTEGER") and c not in ("itemid", "icd_version", "seq_num", "anchor_age", "anchor_year"):
            cols.append(f"TRY_CAST({c} AS {t}) AS {c}")      # measured values: unparseable -> NULL
        else:
            cols.append(f"CAST({c} AS {t}) AS {c}")          # ids / times: fail loudly if malformed
    where = f" WHERE {ROW_FILTERS[(module, table)]}" if (module, table) in ROW_FILTERS else ""
    return f"SELECT * FROM (SELECT {', '.join(cols)} FROM read_csv('{path}', {CSV_OPTS})){where}"


def cached_parquet(module, table, path, select_sql):
    target = CACHE_DIR / f"{module}.{table}.parquet"
    meta = CACHE_DIR / f"{module}.{table}.json"
    stamp = {"source": str(path), "size": path.stat().st_size, "mtime": path.stat().st_mtime,
             "schema": SCHEMA[module][table], "filter": ROW_FILTERS.get((module, table))}
    if target.exists() and meta.exists() and json.loads(meta.read_text()) == stamp:
        return target
    with timed(f"cache {module}.{table}"):
        tmp = target.with_suffix(".parquet.tmp")
        con.execute(f"COPY ({select_sql}) TO '{tmp}' (FORMAT parquet, COMPRESSION zstd)")
        tmp.replace(target)
        meta.write_text(json.dumps(stamp))
    return target


for module, tables in SCHEMA.items():
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {module}")
    for table in tables:
        path = find_csv(module, table)
        select_sql = typed_select(module, table, path)
        if USE_PARQUET_CACHE:
            source = f"SELECT * FROM read_parquet('{cached_parquet(module, table, path, select_sql)}')"
        else:
            source = select_sql
        con.execute(f"CREATE OR REPLACE VIEW {module}.{table} AS {source}")

HOSP = "hosp"
ICU = "icu"


def run(q):
    return con.execute(q).df()


def count(q):
    return int(con.execute(q).fetchone()[0])


run(f"SELECT COUNT(DISTINCT subject_id) AS total_patients FROM {HOSP}.patients")

**Total Patients + Hypertension Patients (Strict vs Broad)**

In [ ]:
run(f"""
CREATE OR REPLACE TEMP TABLE strict_htn AS
SELECT DISTINCT subject_id
FROM {HOSP}.diagnoses_icd
WHERE (icd_version = 10 AND starts_with(icd_code, 'I10'))
   OR (icd_version = 9  AND starts_with(icd_code, '401'))
""")

# Broad definition: this is the one the cohort uses
run(f"""
CREATE OR REPLACE TEMP TABLE htn_patients AS
SELECT DISTINCT subject_id
FROM {HOSP}.diagnoses_icd
WHERE (
        icd_version = 10
        AND regexp_matches(icd_code, '^(I10|I11|I12|I13|I15|I16)')
      )
   OR (
        icd_version = 9
        AND (
            starts_with(icd_code, '401')
            OR starts_with(icd_code, '402')
            OR starts_with(icd_code, '403')
            OR starts_with(icd_code, '404')
            OR starts_with(icd_code, '405')
            OR starts_with(replace(icd_code, '.', ''), '4372')
        )
      )
""")

df_cohort_counts = run(f"""
SELECT
    (SELECT COUNT(DISTINCT subject_id) FROM {HOSP}.patients) AS total_patients,
    (SELECT COUNT(*) FROM strict_htn)   AS strict_htn_patients,
    (SELECT COUNT(*) FROM htn_patients) AS broad_htn_patients
""")
df_cohort_counts

**Defining the HTO MEDICATION Set** (same regex as the Colab notebook)

In [ ]:
HTO_REGEX = r"""
(lisinopril|enalapril|benazepril|captopril|ramipril|
losartan|valsartan|irbesartan|candesartan|olmesartan|
metoprolol|carvedilol|labetalol|atenolol|propranolol|bisoprolol|
amlodipine|nifedipine|diltiazem|verapamil|felodipine|
hydrochlorothiazide|chlorthalidone|indapamide|
furosemide|torsemide|bumetanide|spironolactone|
hydralazine|clonidine|methyldopa|minoxidil)
"""

HTO_REGEX = HTO_REGEX.replace("\n", "")

**LAST HTO index time + every HTO drug ordered at that timestamp** (combination therapy)

In [ ]:
run(f"""
CREATE OR REPLACE TEMP TABLE hto_rx AS
SELECT p.subject_id, p.hadm_id, lower(p.drug) AS drug, p.starttime
FROM {HOSP}.prescriptions p
JOIN htn_patients h USING (subject_id)
WHERE p.starttime IS NOT NULL
  AND p.drug IS NOT NULL
  AND regexp_matches(lower(p.drug), '{HTO_REGEX}')
""")

run("""
CREATE OR REPLACE TEMP TABLE last_index AS
SELECT subject_id, MAX(starttime) AS index_time
FROM hto_rx
GROUP BY subject_id
""")

run("""
CREATE OR REPLACE TEMP TABLE index_treatment AS
SELECT i.subject_id, i.index_time,
       string_agg(DISTINCT h.drug, ' | ' ORDER BY h.drug) AS index_hto_drugs
FROM last_index i
JOIN hto_rx h
  ON i.subject_id = h.subject_id
 AND i.index_time = h.starttime
GROUP BY i.subject_id, i.index_time
""")

print("Patients with an HTO prescription:", count("SELECT COUNT(*) FROM last_index"))
peek(run("SELECT * FROM index_treatment ORDER BY subject_id LIMIT 5"))

**BP outcomes: parsed OMR blood pressure on days 1–90 after the LAST HTO**

In [ ]:
run(f"""
CREATE OR REPLACE TEMP TABLE clean_bp AS
WITH parsed_bp AS (
    SELECT
        subject_id,
        chartdate,
        result_name,
        TRY_CAST(regexp_extract(result_value, '^\\s*(\\d{{2,3}})\\s*/', 1) AS INTEGER) AS sbp,
        TRY_CAST(regexp_extract(result_value, '/\\s*(\\d{{1,3}})\\s*$', 1) AS INTEGER) AS dbp
    FROM {HOSP}.omr
    WHERE lower(result_name) LIKE '%blood pressure%'
)
SELECT *
FROM parsed_bp
WHERE sbp IS NOT NULL AND dbp IS NOT NULL
  AND sbp > 0 AND dbp > 0
  AND sbp > dbp
""")

lo, hi = OUTCOME_WINDOW_DAYS
run(f"""
CREATE OR REPLACE TEMP TABLE outcomes AS
SELECT
    i.subject_id,
    COUNT(b.chartdate)          AS bp_count_90,
    quantile_disc(b.sbp, 0.5)   AS median_sbp_90,
    quantile_disc(b.dbp, 0.5)   AS median_dbp_90,
    MIN(b.chartdate)            AS first_outcome_bp_date,
    MAX(b.chartdate)            AS last_outcome_bp_date
FROM last_index i
LEFT JOIN clean_bp b
  ON i.subject_id = b.subject_id
 AND date_diff('day', CAST(i.index_time AS DATE), b.chartdate) BETWEEN {lo} AND {hi}
GROUP BY i.subject_id
""")

run("SELECT COUNT(*) AS patients, SUM((bp_count_90 > 0)::INT) AS with_outcome_bp FROM outcomes")

**Final LAST-HTO cohort**

Eligibility is the same as in the Colab notebook:
1. Hypertension patient (broad definition)
2. Has an HTO prescription (LAST one = index)
3. Has diagnosis history in an admission with `admittime <= index_time`
4. Has at least one usable BP outcome on days 1–90 after the index

The DX time policy changes which diagnoses become *features*. It doesn't change who is in the cohort, so the cohort matches the earlier one. The exception is `REQUIRE_USABLE_DX = True`, which tightens rule 3 to admissions discharged before the index.

In [ ]:
def dx_history_sql(admission_cond):
    return f"""
    SELECT i.subject_id,
           COUNT(d.icd_code)         AS prior_dx_count,
           COUNT(DISTINCT a.hadm_id) AS prior_visits
    FROM last_index i
    JOIN {HOSP}.admissions a
      ON i.subject_id = a.subject_id
     AND {admission_cond}
    JOIN {HOSP}.diagnoses_icd d
      ON a.hadm_id = d.hadm_id
    GROUP BY i.subject_id
    """

ADMITTED_BEFORE = "a.admittime <= i.index_time"
DISCHARGED_BEFORE = "a.dischtime <= i.index_time"

run("CREATE OR REPLACE TEMP TABLE diagnosis_history AS " +
    dx_history_sql(DISCHARGED_BEFORE if REQUIRE_USABLE_DX else ADMITTED_BEFORE))
run("CREATE OR REPLACE TEMP TABLE completed_dx_history AS " + dx_history_sql(DISCHARGED_BEFORE))

run(f"""
CREATE OR REPLACE TEMP TABLE cohort AS
SELECT
    i.subject_id AS patient_id,
    i.index_time,
    t.index_hto_drugs,
    d.prior_visits,
    d.prior_dx_count,
    o.bp_count_90,
    o.median_sbp_90,
    o.median_dbp_90,
    o.first_outcome_bp_date,
    o.last_outcome_bp_date,
    ((o.median_sbp_90 < {CONTROLLED_SBP}) AND (o.median_dbp_90 < {CONTROLLED_DBP}))::INT AS label
FROM last_index i
JOIN diagnosis_history d USING (subject_id)
JOIN index_treatment t   USING (subject_id)
JOIN outcomes o          USING (subject_id)
WHERE o.bp_count_90 > 0
""")

df_last_cohort = run("SELECT * FROM cohort ORDER BY patient_id")

df_funnel = run("""
SELECT 'broad HTN'                    AS step, COUNT(*) AS patients FROM htn_patients
UNION ALL SELECT '+ HTO prescription',          COUNT(*) FROM last_index
UNION ALL SELECT '+ dx history <= index',       COUNT(*) FROM diagnosis_history
UNION ALL SELECT '+ BP on days 1-90 (cohort)',  COUNT(*) FROM cohort
UNION ALL SELECT '(info) cohort with dx from a completed admission',
                 COUNT(*) FROM cohort c JOIN completed_dx_history h ON c.patient_id = h.subject_id
""")
display(df_funnel)

print("Final LAST-HTO cohort:", len(df_last_cohort))
print(df_last_cohort["label"].value_counts().rename({0: "0 not controlled", 1: "1 controlled"}))
peek(df_last_cohort)

**Sanity check against the BigQuery run** (full data only: a mismatch means the port changed the cohort, so investigate before training)

In [ ]:
ref = REFERENCE.get(DATASET_NAME)
if ref is None:
    print(f"No reference numbers for {DATASET_NAME!r}; skipping.")
elif REQUIRE_USABLE_DX:
    print("REQUIRE_USABLE_DX changes eligibility on purpose; reference cohort size doesn't apply.")
else:
    n, pos = len(df_last_cohort), df_last_cohort["label"].mean()
    rel = abs(n - ref["cohort"]) / ref["cohort"]
    print(f"cohort {n:,} vs reference {ref['cohort']:,} ({rel:.2%} off)")
    print(f"positive rate {pos:.3f} vs reference ~{ref['positive_rate']:.2f}")
    if rel > 0.005 or abs(pos - ref["positive_rate"]) > 0.02:
        print("WARNING: differs from the BigQuery cohort. Check the MIMIC version and the funnel above.")

**Visits** (every admission that starts at or before the index time)

* `is_index_admission`: the index HTO was ordered during this admission (`admittime <= index_time < dischtime`).
* `complete_before_index`: the patient was discharged before the index time.

In [ ]:
run(f"""
CREATE OR REPLACE TEMP TABLE visits AS
SELECT
    c.patient_id,
    a.hadm_id AS visit_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    DENSE_RANK() OVER (PARTITION BY c.patient_id ORDER BY a.admittime, a.hadm_id) AS visit_order,
    (a.admittime <= c.index_time AND a.dischtime > c.index_time) AS is_index_admission,
    (a.dischtime <= c.index_time)                                AS complete_before_index,
    date_diff('day', CAST(a.admittime AS DATE), CAST(c.index_time AS DATE)) AS days_before_index
FROM cohort c
JOIN {HOSP}.admissions a
  ON c.patient_id = a.subject_id
 AND a.admittime <= c.index_time
""")

df_visits = run("SELECT * FROM visits ORDER BY patient_id, visit_order")
print(df_visits.shape)
peek(df_visits, 10)

**Diagnoses**

`time` is when the code becomes available, set by `DX_TIME_POLICY`. Only rows with `time <= index_time` are kept. `diag` is `<version>_<code>` because ICD-9 and ICD-10 codes can collide (both have `V…` codes). `hto-past` turns it into the `D_<diag>` token.

In [ ]:
DX_TIME_COL = "v.dischtime" if DX_TIME_POLICY == "discharge" else "v.admittime"

df_dx_all = run(f"""
SELECT
    v.patient_id,
    v.visit_id,
    v.visit_order,
    v.is_index_admission,
    {DX_TIME_COL}                 AS time,
    c.index_time,
    d.seq_num,
    d.icd_version,
    d.icd_code,
    CAST(d.icd_version AS VARCHAR) || '_' || d.icd_code          AS diag,
    'DX_' || CAST(d.icd_version AS VARCHAR) || '_' || d.icd_code AS token,
    dd.long_title
FROM visits v
JOIN cohort c ON v.patient_id = c.patient_id
JOIN {HOSP}.diagnoses_icd d ON v.visit_id = d.hadm_id
LEFT JOIN {HOSP}.d_icd_diagnoses dd
       ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
ORDER BY v.patient_id, v.visit_order, d.seq_num
""")

usable_mask = df_dx_all["time"] <= df_dx_all["index_time"]
df_dx = df_dx_all[usable_mask].drop(columns="index_time").reset_index(drop=True)

n_excluded = int((~usable_mask).sum())
print(f"Diagnosis rows linked to pre-index admissions: {len(df_dx_all):,}")
print(f"Usable at index time ({DX_TIME_POLICY} policy): {len(df_dx):,}  (excluded {n_excluded:,}, "
      f"all from index admissions: {bool(df_dx_all.loc[~usable_mask, 'is_index_admission'].all())})")
print("Cohort patients left with zero usable diagnoses:",
      len(set(df_last_cohort["patient_id"]) - set(df_dx["patient_id"])))
peek(df_dx, 20)

**Medications** (normalisation copied from the Colab notebook; it runs once per distinct drug name, then the result is joined back in DuckDB)

In [ ]:
def normalize_med_name(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()

    # 1. Remove concentrations / strengths
    text = re.sub(
        r"\b\d+(\.\d+)?\s*"
        r"(mg|mcg|g|ml|meq|units|unit|%)"
        r"(\s*/\s*(ml|l|hr|hour))?\b",
        " ",
        text
    )

    # Handle numbers left before %
    text = re.sub(r"\b\d+(\.\d+)?\s*%", " ", text)

    # 2. Remove formulation / route / packaging words
    text = re.sub(
        r"\b("
        r"tablet|tablets|tab|"
        r"capsule|capsules|cap|"
        r"solution|suspension|"
        r"injection|inj|vial|syringe|ampule|amp|"
        r"oral|po|iv|im|sc|subcutaneous|"
        r"extended release|immediate release|"
        r"delayed release|"
        r"er|xr|xl|sr|dr|"
        r"mini bag plus|mini bag|bag|"
        r"cream|ointment|gel|patch|spray|"
        r"neb|nebulizer|inhalation"
        r")\b",
        " ",
        text
    )

    # 3. Remove punctuation
    text = re.sub(r"[\(\)\[\]\{\},;]", " ", text)
    text = re.sub(r"[\/\\]+", " ", text)

    # 4. Clean whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def canonicalize_med_name(name):
    if not isinstance(name, str):
        return ""

    name = name.strip().lower()

    # common salt/formulation normalization
    replacements = {
        "morphine sulfate": "morphine",
        "docusate sodium": "docusate",
        "levetiracetam": "levetiracetam",
        "sodium chloride": "sodium chloride",

        # beta blocker examples
        "metoprolol tartrate": "metoprolol",
        "metoprolol succinate": "metoprolol",

        # calcium channel blocker example
        "amlodipine besylate": "amlodipine",
    }

    return replacements.get(name, name)


def med_token(normalized):
    return "MED_" + re.sub(r"[^A-Z0-9]+", "_", normalized.upper()).strip("_")

In [ ]:
run(f"""
CREATE OR REPLACE TEMP TABLE meds_raw AS
SELECT
    v.patient_id,
    v.visit_id,
    v.visit_order,
    p.starttime AS time,
    lower(p.drug) AS drug,
    p.route,
    p.ndc
FROM visits v
JOIN {HOSP}.prescriptions p ON v.visit_id = p.hadm_id
JOIN cohort c ON v.patient_id = c.patient_id
WHERE p.drug IS NOT NULL
  AND p.starttime IS NOT NULL
  AND p.starttime <= c.index_time
""")

med_names = run("SELECT DISTINCT drug FROM meds_raw")
med_names["normalized_med"] = med_names["drug"].apply(normalize_med_name).apply(canonicalize_med_name)
med_names["token"] = med_names["normalized_med"].apply(med_token)
con.register("med_name_map", med_names)

# Remove empty names; one row per (patient, visit, token): the earliest order, ties broken by drug name
df_meds_clean = run("""
SELECT patient_id, visit_id, visit_order, time, drug, route, ndc, normalized_med, token
FROM (
    SELECT r.*, m.normalized_med, m.token,
           ROW_NUMBER() OVER (PARTITION BY r.patient_id, r.visit_id, m.token
                              ORDER BY r.time, r.drug, r.route, r.ndc) AS rn
    FROM meds_raw r
    JOIN med_name_map m USING (drug)
    WHERE length(m.normalized_med) > 0
)
WHERE rn = 1
ORDER BY patient_id, visit_order, time, drug
""")

print("Raw medication orders:", count("SELECT COUNT(*) FROM meds_raw"))
print("Raw medication names:", len(med_names))
print("Medication tokens:", df_meds_clean["token"].nunique())
print(df_meds_clean.shape)

Check: every LAST-HTO drug should appear in the medication history, because `starttime <= index_time` includes the index orders.

In [ ]:
def hto_to_tokens(text):
    return {med_token(canonicalize_med_name(normalize_med_name(d.strip()))) for d in str(text).split("|")}

check = df_last_cohort[["patient_id", "index_hto_drugs"]].copy()
check["index_hto_tokens"] = check["index_hto_drugs"].apply(hto_to_tokens)
med_sets = df_meds_clean.groupby("patient_id")["token"].apply(set)
check["hto_present"] = check.apply(
    lambda r: r["index_hto_tokens"].issubset(med_sets.get(r["patient_id"], set())), axis=1
)
print("Percent with all index HTO drugs present:", check["hto_present"].mean() * 100)

**Staging area.** Everything is written to `OUT_DIR/_staging` first. It moves into place only after the leakage checks pass. Labs and vitals go straight from DuckDB to parquet.

In [ ]:
STAGE_DIR = OUT_DIR / "_staging"
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True)


def copy_to_parquet(sql, name):
    path = STAGE_DIR / f"{name}.parquet"
    with timed(f"write {name}"):
        con.execute(f"COPY ({sql}) TO '{path}' (FORMAT parquet, COMPRESSION zstd)")
    return path

**Labs** (numeric lab results charted at or before the index time)

`visit_id` is empty for outpatient labs, and `hto-past` groups those by day. `lab_key` is the `d_labitems` label, which `hto-past`'s multimodal variant matches by name. `itemid`/`fluid` are kept because labels aren't unique across fluids. `LAB_LOOKBACK_DAYS` can cap how far back labs go.

In [ ]:
lookback = f"AND l.charttime >= c.index_time - INTERVAL {LAB_LOOKBACK_DAYS} DAY" if LAB_LOOKBACK_DAYS else ""

labs_path = copy_to_parquet(f"""
SELECT
    c.patient_id,
    l.hadm_id      AS visit_id,
    l.charttime    AS time,
    l.itemid,
    di.label       AS lab_key,
    di.fluid,
    di.category,
    l.valuenum     AS lab_value,
    l.valueuom,
    l.ref_range_lower,
    l.ref_range_upper,
    l.flag
FROM {HOSP}.labevents l
JOIN cohort c ON l.subject_id = c.patient_id
LEFT JOIN {HOSP}.d_labitems di ON l.itemid = di.itemid
WHERE l.valuenum IS NOT NULL
  AND l.charttime <= c.index_time
  {lookback}
""", "final_labs")

print(run(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT patient_id) AS patients, COUNT(DISTINCT itemid) AS lab_items "
          f"FROM read_parquet('{labs_path}')"))
run(f"SELECT lab_key, COUNT(*) AS n FROM read_parquet('{labs_path}') GROUP BY 1 ORDER BY 2 DESC LIMIT 15")

**Vitals**

* OMR (outpatient, date only): BP is split into `bps`/`bpd`, plus `bmi`, `weight_lbs` and `height_in`. Only dates **strictly before** the index date are kept, because a same-day reading could have been taken after the drug was given.
* ICU (optional): NIBP/arterial systolic → `bps`, diastolic → `bpd`, with `charttime <= index_time` and the same plausibility ranges as before.

In [ ]:
omr_sql = f"""
WITH omr AS (
    SELECT o.subject_id AS patient_id, o.chartdate, o.result_name, o.result_value
    FROM {HOSP}.omr o
    JOIN cohort c ON o.subject_id = c.patient_id
    WHERE o.chartdate < CAST(c.index_time AS DATE)
),
bp AS (
    SELECT patient_id, chartdate,
           TRY_CAST(regexp_extract(result_value, '^\\s*(\\d{{2,3}})\\s*/', 1) AS INTEGER) AS sbp,
           TRY_CAST(regexp_extract(result_value, '/\\s*(\\d{{1,3}})\\s*$', 1) AS INTEGER) AS dbp
    FROM omr
    WHERE lower(result_name) LIKE '%blood pressure%'
),
clean_bp AS (
    SELECT * FROM bp WHERE sbp > 0 AND dbp > 0 AND sbp > dbp
)
SELECT patient_id, CAST(NULL AS BIGINT) AS visit_id, CAST(chartdate AS TIMESTAMP) AS time,
       'bps' AS vital_key, CAST(sbp AS DOUBLE) AS vital_value, 'omr' AS source FROM clean_bp
UNION ALL
SELECT patient_id, NULL, CAST(chartdate AS TIMESTAMP), 'bpd', CAST(dbp AS DOUBLE), 'omr' FROM clean_bp
UNION ALL
SELECT patient_id, NULL, CAST(chartdate AS TIMESTAMP),
       CASE WHEN result_name LIKE 'BMI%'    THEN 'bmi'
            WHEN result_name LIKE 'Weight%' THEN 'weight_lbs'
            WHEN result_name LIKE 'Height%' THEN 'height_in' END,
       TRY_CAST(result_value AS DOUBLE), 'omr'
FROM omr
WHERE (result_name LIKE 'BMI%' OR result_name LIKE 'Weight%' OR result_name LIKE 'Height%')
  AND TRY_CAST(result_value AS DOUBLE) IS NOT NULL
"""

icu_sql = f"""
SELECT
    c.patient_id,
    ce.hadm_id   AS visit_id,
    ce.charttime AS time,
    CASE WHEN ce.itemid IN (220179, 220050) THEN 'bps' ELSE 'bpd' END AS vital_key,
    ce.valuenum  AS vital_value,
    'icu'        AS source
FROM {ICU}.chartevents ce
JOIN cohort c ON ce.subject_id = c.patient_id
WHERE ce.itemid IN {ICU_BP_ITEMIDS}
  AND ce.valuenum IS NOT NULL
  AND ce.charttime <= c.index_time
  AND ((ce.itemid IN (220179, 220050) AND ce.valuenum BETWEEN 40 AND 300)
    OR (ce.itemid IN (220180, 220051) AND ce.valuenum BETWEEN 20 AND 200))
"""

vitals_sql = omr_sql + (f"\nUNION ALL\n{icu_sql}" if INCLUDE_ICU_VITALS else "")
vitals_path = copy_to_parquet(vitals_sql, "final_vitals")

run(f"SELECT source, vital_key, COUNT(*) AS rows FROM read_parquet('{vitals_path}') GROUP BY ALL ORDER BY ALL")

**Demographics + stratified train/val/test split**

In [ ]:
df_demo = run(f"""
WITH last_adm AS (
    SELECT a.subject_id, a.race, a.insurance, a.marital_status,
           ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime DESC, a.hadm_id DESC) AS rn
    FROM {HOSP}.admissions a
    JOIN cohort c ON a.subject_id = c.patient_id AND a.admittime <= c.index_time
)
SELECT
    c.patient_id,
    p.gender,
    p.anchor_age + (year(c.index_time) - p.anchor_year) AS age_at_index,
    la.race,
    la.insurance,
    la.marital_status
FROM cohort c
JOIN {HOSP}.patients p ON c.patient_id = p.subject_id
LEFT JOIN last_adm la ON c.patient_id = la.subject_id AND la.rn = 1
""")


def stratified_split(df, seed=SPLIT_SEED, fracs=SPLIT_FRACS):
    rng = np.random.default_rng(seed)
    split = pd.Series(index=df.index, dtype="object")
    for _, idx in sorted(df.groupby("label").groups.items()):
        idx = rng.permutation(np.sort(np.asarray(idx)))
        n_train = int(round(fracs[0] * len(idx)))
        n_val = int(round(fracs[1] * len(idx)))
        split[idx[:n_train]] = "train"
        split[idx[n_train:n_train + n_val]] = "val"
        split[idx[n_train + n_val:]] = "test"
    return split


df_patients = (df_last_cohort.merge(df_demo, on="patient_id", how="left")
               .sort_values("patient_id").reset_index(drop=True))
df_patients["split"] = stratified_split(df_patients)

display(pd.crosstab(df_patients["split"], df_patients["label"], margins=True))

**`hto-past` tables** (the contract in `hto-past/README_RUN_BASELINES.md`)

`hto-past/src/hto/data/build.py` drops events after `latest_bp_time`. In the old pipeline that column was the outcome BP time. Here it holds the **index time**, so the feature cutoff is the index. `bps`/`bpd` are the outcome medians, which `hto-past` only stores and never uses as features.

In [ ]:
final_patients_table = df_patients.assign(
    latest_htn_med_name=df_patients["index_hto_drugs"],
    bps=df_patients["median_sbp_90"].astype(float),
    bpd=df_patients["median_dbp_90"].astype(float),
    latest_bp_time=df_patients["index_time"],   # = feature cutoff used by hto-past build.py
)[[
    "patient_id", "label", "latest_htn_med_name", "bps", "bpd", "latest_bp_time",
    "index_time", "index_hto_drugs", "bp_count_90", "first_outcome_bp_date", "last_outcome_bp_date",
    "prior_visits", "gender", "age_at_index", "race", "insurance", "marital_status", "split",
]]

final_diagnoses = df_dx[[
    "patient_id", "visit_id", "time", "diag", "token", "icd_version", "icd_code",
    "seq_num", "long_title", "is_index_admission",
]]

final_medications = df_meds_clean.rename(columns={"normalized_med": "medication"})[[
    "patient_id", "visit_id", "time", "medication", "token", "drug", "route", "ndc",
]]

final_patients_table.to_parquet(STAGE_DIR / "final_patients_table.parquet", index=False)
final_diagnoses.to_parquet(STAGE_DIR / "final_diagnoses.parquet", index=False)
final_medications.to_parquet(STAGE_DIR / "final_medications.parquet", index=False)

for name in ["final_diagnoses", "final_medications", "final_labs", "final_vitals"]:
    rows, pts = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT patient_id) "
                            f"FROM read_parquet('{STAGE_DIR / (name + '.parquet')}')").fetchone()
    print(f"{name:20s} {rows:>12,} rows  {pts:>7,} patients")

**Flat sequences** (Colab export / `HTO_Eval` format)

Within each visit, diagnoses come first (by `seq_num`), then medications (by start time). Only the most recent `MAX_LEN` tokens are kept. The Colab notebook appended any missing LAST-HTO drugs to the end. That isn't needed here because the index orders are already in the medication history (see the check above).

In [ ]:
dx_events = df_dx.assign(type_order=0, event_order=df_dx["seq_num"].fillna(9999))
med_events = df_meds_clean.assign(
    type_order=1,
    event_order=df_meds_clean.groupby(["patient_id", "visit_id"]).cumcount(),
)
cols = ["patient_id", "visit_order", "type_order", "event_order", "token"]
all_events = (
    pd.concat([dx_events[cols], med_events[cols]], ignore_index=True)
    .sort_values(["patient_id", "visit_order", "type_order", "event_order"], kind="stable")
)

df_sequences = all_events.groupby("patient_id")["token"].apply(list).reset_index(name="sequence_full")
df_sequences = df_patients[["patient_id", "label"]].merge(df_sequences, on="patient_id", how="left")
df_sequences["sequence_full"] = df_sequences["sequence_full"].apply(lambda s: s if isinstance(s, list) else [])
df_sequences["sequence"] = df_sequences["sequence_full"].apply(lambda s: s[-MAX_LEN:])
df_sequences["sequence_length"] = df_sequences["sequence_full"].apply(len)

df_sequences[["patient_id", "sequence"]].to_parquet(STAGE_DIR / "final_sequences.parquet", index=False)

print(df_sequences["sequence_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"Patients truncated (> {MAX_LEN}):", (df_sequences["sequence_length"] > MAX_LEN).sum())

**ICD inventory** (input for the ICD → SNOMED CT mapping)

Each distinct diagnosis code in the cohort, with a dotted form for matching against the SNOMED ICD-10-CM map and the ICD-9 crosswalks. It also records how many patients have it as a usable feature vs. at all.

In [ ]:
def icd_dotted(code, version):
    code = str(code).upper()
    if version == 10:
        return code if len(code) <= 3 else f"{code[:3]}.{code[3:]}"
    if code.startswith("E"):
        return code if len(code) <= 4 else f"{code[:4]}.{code[4:]}"
    return code if len(code) <= 3 else f"{code[:3]}.{code[3:]}"


usable = df_dx.groupby(["icd_version", "icd_code"]).agg(
    n_rows_usable=("patient_id", "size"), n_patients_usable=("patient_id", "nunique"))
anyuse = df_dx_all.groupby(["icd_version", "icd_code"]).agg(
    long_title=("long_title", "first"), n_patients_any=("patient_id", "nunique"))

icd_inventory = anyuse.join(usable, how="left").fillna({"n_rows_usable": 0, "n_patients_usable": 0}).reset_index()
icd_inventory["n_rows_usable"] = icd_inventory["n_rows_usable"].astype(int)
icd_inventory["n_patients_usable"] = icd_inventory["n_patients_usable"].astype(int)
icd_inventory["icd_dotted"] = [icd_dotted(c, v) for c, v in zip(icd_inventory["icd_code"], icd_inventory["icd_version"])]
icd_inventory["token"] = "DX_" + icd_inventory["icd_version"].astype(str) + "_" + icd_inventory["icd_code"]
icd_inventory = icd_inventory.sort_values(["n_patients_usable", "n_patients_any", "token"],
                                          ascending=[False, False, True])[[
    "token", "icd_version", "icd_code", "icd_dotted", "long_title",
    "n_patients_usable", "n_rows_usable", "n_patients_any",
]]
icd_inventory.to_csv(STAGE_DIR / "icd_inventory.csv", index=False)

display(icd_inventory.groupby("icd_version").agg(codes=("token", "size"), usable_rows=("n_rows_usable", "sum")))
display(icd_inventory.head(15))

**Leakage checks** (run in DuckDB on the staged files; all must pass before anything is moved into place)

In [ ]:
def staged(name):
    return f"read_parquet('{STAGE_DIR / (name + '.parquet')}')"


checks = {}
for name in ["final_diagnoses", "final_medications", "final_labs", "final_vitals"]:
    late = count(f"SELECT COUNT(*) FROM {staged(name)} e JOIN cohort c USING (patient_id) WHERE e.time > c.index_time")
    orphan = count(f"SELECT COUNT(*) FROM {staged(name)} e ANTI JOIN cohort c USING (patient_id)")
    checks[f"{name}: no event after index_time"] = late == 0
    checks[f"{name}: only cohort patients"] = orphan == 0

checks["final_vitals: OMR strictly before index date"] = count(f"""
    SELECT COUNT(*) FROM {staged('final_vitals')} e JOIN cohort c USING (patient_id)
    WHERE e.source = 'omr' AND CAST(e.time AS DATE) >= CAST(c.index_time AS DATE)""") == 0
checks["outcome BP starts >= 1 day after index"] = count(f"""
    SELECT COUNT(*) FROM cohort
    WHERE date_diff('day', CAST(index_time AS DATE), first_outcome_bp_date) < {OUTCOME_WINDOW_DAYS[0]}""") == 0
if DX_TIME_POLICY == "discharge":
    checks["final_diagnoses: no index-admission codes"] = not bool(final_diagnoses["is_index_admission"].any())
checks["every patient has exactly one split"] = bool(final_patients_table["split"].notna().all()
                                                     and final_patients_table["patient_id"].is_unique)

for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
assert all(checks.values()), "Leakage check failed: outputs left in _staging, nothing moved into place"

**Exporting the Data**

```
data/processed/<dataset>/
  mimic_data/              ← hto-past input contract (symlink or copy into hto-past/mimic_data/)
    final_patients_table.parquet
    final_diagnoses.parquet
    final_medications.parquet
    final_labs.parquet
    final_vitals.parquet
    final_sequences.parquet
  tables/
    visits.parquet         ← admission timeline (KerPrint τ, visit grouping)
    icd_inventory.csv      ← codes to map to SNOMED CT
    splits.csv
  hto_eval/
    bp_patient_sequences.csv   ← HTO_Eval format (patient_id, sequence JSON, label)
  figures/
  manifest.json            ← config, funnel, row counts, checks
  DATA_DICTIONARY.md
```

In [ ]:
for sub in ["mimic_data", "tables", "hto_eval", "figures"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

for name in ["final_patients_table", "final_diagnoses", "final_medications",
             "final_labs", "final_vitals", "final_sequences"]:
    (STAGE_DIR / f"{name}.parquet").replace(OUT_DIR / "mimic_data" / f"{name}.parquet")
(STAGE_DIR / "icd_inventory.csv").replace(OUT_DIR / "tables" / "icd_inventory.csv")
STAGE_DIR.rmdir()

df_visits.to_parquet(OUT_DIR / "tables/visits.parquet", index=False)
final_patients_table[["patient_id", "label", "split"]].to_csv(OUT_DIR / "tables/splits.csv", index=False)

model_data = df_sequences[["patient_id", "sequence", "label"]].copy()
model_data["sequence"] = model_data["sequence"].apply(json.dumps)
model_data.to_csv(OUT_DIR / "hto_eval/bp_patient_sequences.csv", index=False)


def parquet_rows(name):
    return count(f"SELECT COUNT(*) FROM read_parquet('{OUT_DIR / 'mimic_data' / (name + '.parquet')}')")


manifest = {
    "dataset": DATASET_NAME,
    "raw_dir": str(RAW_DIR),
    "created": dt.datetime.now().isoformat(timespec="seconds"),
    "versions": {"duckdb": duckdb.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "config": {
        "outcome_window_days": list(OUTCOME_WINDOW_DAYS),
        "controlled_if": f"median_sbp_90 < {CONTROLLED_SBP} and median_dbp_90 < {CONTROLLED_DBP}",
        "dx_time_policy": DX_TIME_POLICY,
        "require_usable_dx": REQUIRE_USABLE_DX,
        "include_icu_vitals": INCLUDE_ICU_VITALS,
        "lab_lookback_days": LAB_LOOKBACK_DAYS,
        "max_len": MAX_LEN,
        "split_seed": SPLIT_SEED,
        "split_fracs": list(SPLIT_FRACS),
    },
    "funnel": dict(zip(df_funnel["step"], df_funnel["patients"].astype(int))),
    "label_counts": {str(k): int(v) for k, v in final_patients_table["label"].value_counts().sort_index().items()},
    "split_counts": {k: int(v) for k, v in final_patients_table["split"].value_counts().sort_index().items()},
    "rows": {
        **{name: parquet_rows(name) for name in ["final_diagnoses", "final_medications", "final_labs", "final_vitals"]},
        "visits": len(df_visits),
        "icd_codes": len(icd_inventory),
    },
    "dx_rows_excluded_by_time_policy": n_excluded,
    "patients_with_zero_usable_dx": len(set(df_last_cohort["patient_id"]) - set(df_dx["patient_id"])),
    "checks": checks,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps({k: manifest[k] for k in ["funnel", "label_counts", "split_counts", "rows"]}, indent=2))

In [ ]:
DATA_DICTIONARY = """# Data dictionary: {dataset}

Produced by `extraction/MIMIC_IV_Extraction.ipynb`. All event times are MIMIC's shifted timestamps. Every feature row satisfies `time <= index_time`.
This folder contains MIMIC-IV-derived data: share it only with people covered by the PhysioNet data use agreement.

## mimic_data/final_patients_table.parquet (one row per patient)
| column | meaning |
|---|---|
| patient_id | MIMIC subject_id |
| label | 1 = controlled (median SBP < {sbp} and median DBP < {dbp} on days {lo}-{hi} after index) |
| index_time | start time of the LAST HTO prescription (the feature cutoff) |
| latest_bp_time | = index_time; the column name hto-past's build.py uses as its cutoff |
| latest_htn_med_name / index_hto_drugs | every HTO drug ordered at index_time, joined with ' \\| ' |
| bps, bpd | outcome median SBP/DBP (label inputs only; never use as features) |
| bp_count_90, first/last_outcome_bp_date | outcome BP readings in the window |
| prior_visits | admissions counted by the eligibility rule |
| gender, age_at_index, race, insurance, marital_status | demographics (race etc. from the latest pre-index admission) |
| split | train / val / test (stratified by label, seed {seed}) |

## mimic_data/final_diagnoses.parquet
`patient_id, visit_id (hadm_id), time, diag, token, icd_version, icd_code, seq_num, long_title, is_index_admission`
`time` = {dx_time} (DX_TIME_POLICY = "{policy}"). `diag` = `<version>_<code>`; hto-past makes `D_<diag>`.

## mimic_data/final_medications.parquet
`patient_id, visit_id, time (starttime), medication (normalised), token (MED_*), drug (raw), route, ndc`. One row per visit per token.

## mimic_data/final_labs.parquet
`patient_id, visit_id (null = outpatient), time, itemid, lab_key (d_labitems.label), fluid, category, lab_value, valueuom, ref_range_lower/upper, flag`

## mimic_data/final_vitals.parquet
`patient_id, visit_id (null for OMR), time, vital_key (bps, bpd, bmi, weight_lbs, height_in), vital_value, source (omr / icu)`. OMR rows are dated strictly before the index date.

## mimic_data/final_sequences.parquet, hto_eval/bp_patient_sequences.csv
Flat DX/MED token sequence per patient (most recent {max_len}), in the Colab export format.

## tables/visits.parquet
Admissions starting at or before the index time: `visit_id, admittime, dischtime, visit_order, is_index_admission, complete_before_index, days_before_index`.

## tables/icd_inventory.csv
Distinct cohort diagnosis codes, with dotted form, title and patient counts (usable vs any). This is the input for ICD → SNOMED mapping.
""".format(dataset=DATASET_NAME, sbp=CONTROLLED_SBP, dbp=CONTROLLED_DBP, lo=OUTCOME_WINDOW_DAYS[0],
           hi=OUTCOME_WINDOW_DAYS[1], seed=SPLIT_SEED, max_len=MAX_LEN, policy=DX_TIME_POLICY,
           dx_time="admission dischtime" if DX_TIME_POLICY == "discharge" else "admission admittime")

(OUT_DIR / "DATA_DICTIONARY.md").write_text(DATA_DICTIONARY)
print("Wrote", OUT_DIR)

### **Graphs**

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

label_counts = final_patients_table["label"].value_counts().sort_index()
bars = axes[0].bar(["Label 0\nNot controlled", "Label 1\nControlled"], label_counts.values)
for bar, n in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{n:,}\n({n / label_counts.sum() * 100:.1f}%)", ha="center", va="bottom")
axes[0].set_title("Outcome label distribution")
axes[0].set_ylabel("Patients")

visits_per_patient = df_visits.groupby("patient_id").size()
axes[1].hist(visits_per_patient, bins=range(1, min(visits_per_patient.max(), 50) + 2), edgecolor="black")
axes[1].set_title("Admissions up to index per patient")
axes[1].set_xlabel("Admissions (capped at 50 for display)")

axes[2].hist(df_sequences["sequence_length"].clip(upper=1500), bins=50, edgecolor="black")
axes[2].axvline(MAX_LEN, linestyle="--", label=f"{MAX_LEN}-token cutoff")
axes[2].set_title("Flat sequence length (DX + MED)")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "figures/cohort_overview.png", dpi=200, bbox_inches="tight")
plt.show()